# The Unofficial Guide

This notebook turns the 10 raw text files in `data/raw` into a small retrieval system for Harvard CS student knowledge. It cleans the documents, chunks them, builds a lightweight vector index, answers questions with cited evidence, and prints an evaluation report.

One source file is empty, so the pipeline records it but does not create chunks from it.

In [ ]:
from collections import Counter
from pathlib import Path
import math
import re

ROOT = Path.cwd()
RAW_DIR = ROOT / 'data' / 'raw'

STOPWORDS = {
    'a', 'an', 'and', 'are', 'as', 'at', 'be', 'but', 'by', 'can', 'could',
    'did', 'do', 'does', 'for', 'from', 'had', 'has', 'have', 'he', 'her',
    'here', 'him', 'his', 'how', 'i', 'if', 'in', 'into', 'is', 'it', 'its',
    'just', 'me', 'more', 'my', 'not', 'of', 'on', 'or', 'our', 'out', 'so',
    'than', 'that', 'the', 'their', 'them', 'then', 'there', 'these', 'they',
    'this', 'those', 'to', 'up', 'was', 'we', 'were', 'what', 'when', 'where',
    'which', 'who', 'why', 'will', 'with', 'would', 'you', 'your'
}

AD_LINE_PATTERNS = (
    'promoted',
    'learn more',
    'shop now',
    'clickable image',
    'collapse video player',
    '0:00 / 0:00',
    'docs.qvac',
    'adobe.com',
    'gemini.google.com',
    'avatar',
    'comments section'
)

def preprocess_text(text: str) -> str:
    cleaned_lines = []
    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line:
            continue
        lower = line.lower()
        if any(pattern in lower for pattern in AD_LINE_PATTERNS):
            continue
        if re.fullmatch(r'[0-9·•\s:/]+', line):
            continue
        if lower in {'read', 'sort'}:
            continue
        cleaned_lines.append(line)
    cleaned_text = ' '.join(cleaned_lines)
    cleaned_text = re.sub(r'\s+', ' ', cleaned_text).strip()
    return cleaned_text

def chunk_text(text: str, chunk_size: int = 120, overlap: int = 30):
    words = text.split()
    if not words:
        return []
    chunks = []
    start = 0
    chunk_index = 1
    while start < len(words):
        end = min(len(words), start + chunk_size)
        chunk_words = words[start:end]
        chunks.append({
            'chunk_index': chunk_index,
            'start_word': start,
            'end_word': end,
            'text': ' '.join(chunk_words),
        })
        if end >= len(words):
            break
        start = end - overlap
        chunk_index += 1
    return chunks

def load_documents(raw_dir=RAW_DIR):
    documents = []
    for path in sorted(raw_dir.glob('*.txt')):
        raw_text = path.read_text(encoding='utf-8', errors='ignore')
        clean_text = preprocess_text(raw_text)
        documents.append({
            'source': path.name,
            'title': path.stem.replace('_', ' '),
            'raw_text': raw_text,
            'clean_text': clean_text,
            'raw_characters': len(raw_text),
            'clean_characters': len(clean_text),
        })
    return documents

documents = load_documents()
chunks = []
for document in documents:
    for chunk in chunk_text(document['clean_text']):
        chunks.append({
            'source': document['source'],
            'title': document['title'],
            'chunk_index': chunk['chunk_index'],
            'chunk_id': f"{document['source']}::chunk-{chunk['chunk_index']}",
            'search_text': f"{document['title']} {chunk['text']}",
            'text': chunk['text'],
        })

print(f'Loaded {len(documents)} files from data/raw.')
print(f"Non-empty files: {sum(1 for document in documents if document['clean_text'])}")
print(f'Total chunks: {len(chunks)}')
for document in documents:
    print(f"- {document['source']}: {document['clean_characters']} cleaned characters")

In [ ]:
def tokenize(text: str):
    tokens = re.findall(r"[a-z0-9']+", text.lower())
    return [token for token in tokens if token not in STOPWORDS]

SYNONYM_MAP = {
    'beginner': ['beginners', 'beginner-friendly', 'new', 'newbie'],
    'professor': ['professors', 'faculty', 'instructor', 'teacher'],
    'ranking': ['ranked', 'rank', 'rankings', 'recruiting'],
    'certificate': ['certification', 'certificate', 'useful'],
    'experience': ['experience', 'student experience', 'undergraduate experience'],
    'course': ['course', 'class', 'classes', 'cs50', 'cs50p'],
    'research': ['research', 'academic', 'academia'],
}

def expand_query(query: str) -> str:
    lower_query = query.lower()
    additions = []
    for key, terms in SYNONYM_MAP.items():
        if key in lower_query:
            additions.extend(terms)
    return query + ' ' + ' '.join(additions)

def build_tfidf_index(chunks):
    tokenized_chunks = [tokenize(chunk.get('search_text', chunk['text'])) for chunk in chunks]
    document_frequency = Counter()
    for tokens in tokenized_chunks:
        document_frequency.update(set(tokens))

    total_documents = len(chunks) or 1
    idf = {
        term: math.log((1 + total_documents) / (1 + count)) + 1
        for term, count in document_frequency.items()
    }

    vectors = []
    norms = []
    for tokens in tokenized_chunks:
        tf = Counter(tokens)
        total_tokens = len(tokens) or 1
        vector = {
            term: (count / total_tokens) * idf[term]
            for term, count in tf.items()
        }
        norm = math.sqrt(sum(value * value for value in vector.values())) or 1.0
        vectors.append(vector)
        norms.append(norm)
    return idf, vectors, norms

idf, chunk_vectors, chunk_norms = build_tfidf_index(chunks)

def cosine_similarity(left_vector, left_norm, right_vector, right_norm):
    if not left_vector or not right_vector:
        return 0.0
    dot_product = 0.0
    if len(left_vector) > len(right_vector):
        left_vector, right_vector = right_vector, left_vector
    for term, value in left_vector.items():
        dot_product += value * right_vector.get(term, 0.0)
    return dot_product / (left_norm * right_norm)

def vectorize_query(query: str):
    query_tokens = tokenize(expand_query(query))
    tf = Counter(query_tokens)
    total_tokens = len(query_tokens) or 1
    vector = {
        term: (count / total_tokens) * idf[term]
        for term, count in tf.items()
        if term in idf
    }
    norm = math.sqrt(sum(value * value for value in vector.values())) or 1.0
    return vector, norm

def search(query: str, top_k: int = 3):
    query_vector, query_norm = vectorize_query(query)
    scored_chunks = []
    for chunk, chunk_vector, chunk_norm in zip(chunks, chunk_vectors, chunk_norms):
        score = cosine_similarity(query_vector, query_norm, chunk_vector, chunk_norm)
        scored_chunks.append({**chunk, 'score': score})
    scored_chunks.sort(key=lambda item: item['score'], reverse=True)
    return scored_chunks[:top_k]

In [ ]:
def split_sentences(text: str):
    sentences = re.split(r'(?<=[.!?])\s+', text.strip())
    return [sentence.strip() for sentence in sentences if sentence.strip()]

def generate_answer(query: str, top_k: int = 3):
    retrieved = search(query, top_k=top_k)
    if not retrieved or retrieved[0]['score'] == 0:
        return {
            'answer': 'I could not find enough evidence in the retrieved documents to answer that clearly.',
            'sources': [],
            'retrieved': retrieved,
        }

    query_tokens = set(tokenize(expand_query(query)))
    candidate_sentences = []
    for item in retrieved:
        for sentence in split_sentences(item['text']):
            sentence_tokens = set(tokenize(sentence))
            overlap = len(query_tokens & sentence_tokens)
            if overlap:
                candidate_sentences.append((overlap, len(sentence), sentence, item))

    candidate_sentences.sort(key=lambda item: (item[0], -item[1]), reverse=True)
    selected_sentences = []
    sources = []
    seen_sentences = set()
    for _, _, sentence, item in candidate_sentences:
        key = sentence.lower()
        if key in seen_sentences:
            continue
        seen_sentences.add(key)
        selected_sentences.append(sentence)
        source_label = f"{item['source']} [chunk {item['chunk_index']}]"
        if source_label not in sources:
            sources.append(source_label)
        if len(selected_sentences) == 3:
            break

    if not selected_sentences:
        selected_sentences = [retrieved[0]['text'][:300].rstrip()]
        sources = [f"{retrieved[0]['source']} [chunk {retrieved[0]['chunk_index']}]"]

    answer = 'Based on the retrieved documents, ' + ' '.join(selected_sentences)
    return {
        'answer': answer,
        'sources': sources,
        'retrieved': retrieved,
    }

def ask(query: str, top_k: int = 3):
    result = generate_answer(query, top_k=top_k)
    print(f'Query: {query}')
    print()
    print(result['answer'])
    print()
    print('Sources:')
    if result['sources']:
        for source in result['sources']:
            print(f'- {source}')
    else:
        print('- None')
    print()
    print('Top retrieved chunks:')
    for rank, item in enumerate(result['retrieved'], start=1):
        snippet = item['text'][:220].replace('\n', ' ')
        print(f"{rank}. {item['source']} [chunk {item['chunk_index']}] score={item['score']:.3f}")
        print(f'   {snippet}...')
    return result

demo_queries = [
    'Is CS50 beginner friendly?',
    'Is the CS50 certificate useful?',
    'How is the Harvard CS department overall?',
]

for query in demo_queries:
    print('=' * 90)
    ask(query)

## Evaluation harness

The evaluation set below uses five questions with ground-truth answers derived from the source files. One question is intentionally harder because it asks for a numeric conversion rather than a direct quote.

In [ ]:
evaluation_set = [
    {
        'question': 'Is CS50 beginner friendly?',
        'gold_answer': 'Mostly yes. The comments say it is meant for beginners, but some problem sets are hard; CS50P is easier for people who want a gentler Python-only starting point.',
        'expected_sources': ['Harvard_CS50.txt'],
        'retrieval_expected': 'accurate',
        'response_expected': 'accurate',
    },
    {
        'question': 'Is the CS50 certificate useful?',
        'gold_answer': 'Not really for the course discussion here. The documents say CS50 is free and that the certificate is not the main reason to take it; the value is the learning.',
        'expected_sources': ['Harvard_CS50.txt'],
        'retrieval_expected': 'accurate',
        'response_expected': 'accurate',
    },
    {
        'question': 'How is the Harvard CS department overall?',
        'gold_answer': 'It is small compared with MIT and CMU, but people describe it as having top-notch faculty, strong research access, and plenty of growth.',
        'expected_sources': ['Harvard_Experience.txt'],
        'retrieval_expected': 'accurate',
        'response_expected': 'accurate',
    },
    {
        'question': 'Are Harvard CS professors approachable?',
        'gold_answer': 'Yes. The comments describe professors as approachable and mention Madhu Sudan as an example, including advising groups and office hours.',
        'expected_sources': ['Harvard_Experience.txt'],
        'retrieval_expected': 'accurate',
        'response_expected': 'accurate',
    },
    {
        'question': 'Approximately what percentage of Harvard CS faculty hold or have held industry positions?',
        'gold_answer': 'The article says 12 of 43 SEAS tenure-track professors affiliated with CS hold or have held industry positions. That is about 28 percent.',
        'expected_sources': ['Harvard_CS_Faculty.txt'],
        'retrieval_expected': 'accurate',
        'response_expected': 'partially accurate',
    },
]

evaluation_results = []
for item in evaluation_set:
    result = generate_answer(item['question'], top_k=3)
    evaluation_results.append({
        'question': item['question'],
        'gold_answer': item['gold_answer'],
        'system_answer': result['answer'],
        'retrieved_chunks': [
            f"{hit['source']} [chunk {hit['chunk_index']}]"
            for hit in result['retrieved']
        ],
        'retrieval_expected': item['retrieval_expected'],
        'response_expected': item['response_expected'],
        'sources': result['sources'],
    })

for index, row in enumerate(evaluation_results, start=1):
    print(f"Q{index}: {row['question']}")
    print(f"Gold: {row['gold_answer']}")
    print(f"System: {row['system_answer']}")
    print(f"Retrieved: {', '.join(row['retrieved_chunks'])}")
    print(f"Retrieval: {row['retrieval_expected']} | Response: {row['response_expected']}")
    print()